# Lab 08: Compiled code and Python interop

**We'll speed up a calculation, then check that it still gives the right answer.**

We'll use the same small problem in Python, NumPy, Numba, and C++. Then we'll look at how Julia fits into this workflow. The useful part is being able to move one expensive calculation out of Python while keeping the rest of your notebook readable.

[Setup](#setup) · [A small calculation](#a-small-calculation) · [NumPy](#numpy-already-uses-compiled-code) · [Numba](#compile-the-loop-with-numba) · [Timing](#time-the-same-work) · [C++](#call-c-from-python) · [Julia](#where-julia-fits) · [Your turn](#your-turn) · [Reference](#quick-reference)

<a id="setup"></a>

## Setup

Use a **Python 3** notebook on DataHub or Colab. Run the cells in order. The C++ section also needs a Linux runtime with `g++`; Julia is a read-through with an optional example to try later.

If an import below is missing, run this in a new code cell, then restart the kernel:

```python
%pip install numpy matplotlib numba
```

In [ ]:
import sys
import random
from math import sqrt
from statistics import median
from time import perf_counter
from timeit import repeat

print("Python:", sys.version.split()[0])

**Where is the work happening?**

| What we write | What runs the numerical work |
| --- | --- |
| A regular Python loop | Python handles each iteration. |
| A NumPy array expression | NumPy calls compiled routines. |
| A function decorated with `@njit` | Numba compiles supported numerical Python when we first call it. |
| A C++ function called from Python | We compile a library first, then Python calls it. |
| A Julia function | Julia compiles methods for the argument types it receives. |

You don't have to change languages to use compiled code. If you're using NumPy, you already are.

<a id="a-small-calculation"></a>

## A small calculation

**Start with something we can check by hand.**

Take points uniformly distributed in the unit square. Count how many satisfy `x*x + y*y < 1`. That fraction estimates the area of a quarter circle, so four times the fraction estimates π.

We'll keep the random-number generation outside the function. Each version gets the **same points**. Otherwise a change in the answer could just come from drawing a different sample.

We'll start with ordinary Python: a list of `(x, y)` pairs, a loop, and the standard library. Points exactly on the circle are excluded by our choice of `<`.

In [ ]:
## remember (...) are tuples and [...] are lists. what's the functional difference? i forgot.
# but it's useful as an organizational aid.

small_python = [(0.0, 0.0), (0.3, 0.4), (1.0, 0.0), (1.0, 1.0)]

In [ ]:
def count_python(points):    # takes in a list of tuples (points)
    count = 0                # initializes a counter
    for x, y in points:      # initializes iteration variables. python syntax is pretty smart and can just
        if x*x + y*y < 1.0:
            count += 1
    return count

In [ ]:
print(f'num points in unit quarter circle: {count_python(small_python)}')

**Which two points count?** Check that before running the larger example.

In [ ]:
seed = 142
rng = random.Random(seed) # the way we do random numbers in vanilla python

# make a list of random points with values between 0 and 1.
points_python = []
for i in range(100_000): # python interprets 100_000 as 100000, just easier to read.
    x = rng.random()
    y = rng.random()
    points_python.append((x, y))

# Syntax shortcut: use this instead of the loop above.
# points_python = [(rng.random(), rng.random()) for _ in range(100_000)]

<details>
<summary><strong>Advanced Python lists: list comprehensions</strong></summary>

<p>That commented line is a <strong>list comprehension</strong>: a shortcut for creating a list with a loop.</p>

<pre><code>[number * number for number in range(5)]
# Gives [0, 1, 4, 9, 16]</code></pre>

<p>Read it as: “for each number, calculate its square and put it in the list.” The expression that makes each entry comes first.</p>

<p>In our commented shortcut, <code>(rng.random(), rng.random())</code> makes one coordinate pair. <code>for _ in range(100_000)</code> repeats that 100,000 times. The <code>_</code> means we don't need the loop counter. The outer square brackets collect the pairs into a list.</p>

<p>Both versions draw two random numbers per point. If you want to compare their lists exactly, reset <code>rng = random.Random(seed)</code> before each version. Use whichever form makes your code easier to read.</p>

</details>

In [ ]:
print(f'first ten out of 100,000 points generated: \n{points_python[:10]}')

In [ ]:
count = count_python(points_python)
p_inside = count / len(points_python)
pi_estimate = 4 * p_inside
pi_stderr = 4 * sqrt(p_inside * (1 - p_inside) / len(points_python))
print(f"seed = {seed}, points = {len(points_python):,}")
print(f"pi estimate = {pi_estimate:.5f} +/- {pi_stderr:.5f}")

That uncertainty is an estimated **standard error** from counting independent hits and misses. It describes the sampling noise in this example. It isn't an error formula to copy into a correlated simulation.

An estimate consistent with π within its sampling uncertainty is a useful check of the calculation. It does **not** prove that two implementations agree. For that, compare their counts on exactly the same points.

<a id="numpy-already-uses-compiled-code"></a>

## NumPy already uses compiled code

**Write the same calculation with arrays.**

Now we'll import NumPy and convert the lists into arrays. These are the **same coordinates**, not a new random sample. We keep the original lists for the Python timing.

In [ ]:
import numpy as np

points = np.array(points_python, dtype=np.float64)
small = np.array(small_python, dtype=np.float64)
print("NumPy:", np.__version__)
print("Array shape:", points.shape, "| dtype:", points.dtype)

`count_nonzero` counts the `True` entries in the condition below. There is no Python loop over individual points.

In [ ]:
def count_numpy(points):
    radius_squared = points[:, 0]**2 + points[:, 1]**2
    return np.count_nonzero(radius_squared < 1.0)

assert count_numpy(small) == 2
assert count_numpy(points) == count_python(points_python)

This is short and readable. It also creates intermediate arrays for the squares, sum, and comparison. That can matter for larger inputs. A compiled loop can do the arithmetic and counting in one pass without keeping those arrays.

That doesn't tell us which version is faster on this machine. We still need to measure it.

**Here's what we're counting.** We'll use NumPy and Matplotlib for this picture. Plotting stays outside the calculation we're comparing.

In [ ]:
import matplotlib.pyplot as plt

shown = points[:1_500]
inside = shown[:, 0]**2 + shown[:, 1]**2 < 1
theta = np.linspace(0, np.pi / 2, 200)

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(*shown[inside].T, s=8, label="Counted")
ax.scatter(*shown[~inside].T, s=8, marker="x", label="Not counted")
ax.plot(np.cos(theta), np.sin(theta), color="black")
ax.set(xlabel="x", ylabel="y", xlim=(0, 1), ylim=(0, 1),
       aspect="equal", title="circle_sample_v2: first 1,500 points")
ax.legend()
plt.show()

<a id="compile-the-loop-with-numba"></a>

## Compile the loop with Numba

**Same loop. This time, compile it for the array.**

The loop below has the same arithmetic and condition as `count_python`. We give it the NumPy array and add `@njit`, which asks Numba to compile the function into machine code. Numba works well with numerical loops and arrays. It won't compile an arbitrary notebook: keep plotting, file handling, and setup outside the numerical function.

In [ ]:
import numba
from numba import njit

print("Numba:", numba.__version__)

@njit
def count_numba(points):
    count = 0
    for x, y in points:
        if x*x + y*y < 1.0:
            count += 1
    return count

**The first call includes compilation.**

Run the next cell once before timing repeated calls. Rerunning it in the same kernel usually measures an already-compiled function. Restart the kernel if you want to measure a fresh first call again.

In [ ]:
start = perf_counter()
numba_count = count_numba(points)
first_call_seconds = perf_counter() - start

assert numba_count == count
assert count_numba(small) == 2
print(f"First call: {first_call_seconds:.4f} s")
print("Counts agree:", numba_count)

Numba specializes on argument types. A different array dtype can trigger another compilation; changing only its length usually doesn't.

For later work, pass changing parameters as arguments. Don't hide them in a global variable and assume compiled code will notice every edit. Start with ordinary `@njit`; options such as `fastmath=True` can change floating-point behavior.

<a id="time-the-same-work"></a>

## Time the same work

**Keep setup and plotting out of this comparison.**

The function below measures five batches of three calls and reports the median time per call. Python gets the original **list**; NumPy and Numba get the **array of the same points**. Conversion is already done, and Numba has been compiled for this array type.

We're comparing these implementations, including their choice of storage. The first-call and conversion costs still matter for a short script.

In [ ]:
def median_seconds(function, data):
    batches = repeat(lambda: function(data), repeat=5, number=3)
    return median(batches) / 3

timings = {
    "Python": median_seconds(count_python, points_python),
    "NumPy": median_seconds(count_numpy, points),
    "Numba": median_seconds(count_numba, points),
}

for name, seconds in timings.items():
    speedup = timings["Python"] / seconds
    print(f"{name:8s} {seconds * 1000:9.3f} ms   {speedup:7.1f}x")

**What did we leave out?** Generating the points, converting the list to an array, imports, compilation, and the plot. That's useful for comparing the counters. To find out whether your whole program got faster, time the whole calculation too.

A faster counter doesn't make a fixed sample more accurate. It may let you afford more samples. In a correlated simulation, more updates also don't automatically mean more independent measurements.

### Before using this in your homework

- **Find the slow part.** Time construction, repeated numerical work, and plotting separately. A slow animation doesn't imply a slow simulation.
- **Keep library operations where they belong.** NumPy matrix products and SciPy solvers already call compiled routines. Wrapping them in a decorator isn't automatically an improvement.
- **Keep the calculation the same.** Changing the grid, dtype, algorithm, or number of samples changes what you're comparing.
- **Check a small case first.** Matching two implementations catches translation errors. You still need a separate physics check and a convergence check.

For a calculation that repeats an update, keep the repeated work inside the numerical function when practical. Calling across the Python/compiled boundary for every scalar operation can eat up the gain.

### A small dtype check

**Complex numbers need complex storage.**

Before speeding up anything that uses complex arrays, try a tiny example with a nonzero imaginary part. Here's a rotation of a complex number. Its magnitude should stay the same.

In [ ]:
@njit
def rotate(z, angle):
    return z * np.exp(1j * angle)

z = np.array([1 + 2j, -3j], dtype=np.complex128)
rotated = rotate(z, 0.4)
np.testing.assert_allclose(np.abs(rotated), np.abs(z),
                           rtol=1e-12, atol=1e-12)
print("Input:", z.dtype, "| Output:", rotated.dtype)

`assert_allclose` allows small rounding differences. Choose tolerances for the scale of your calculation; these values are for this tiny check. For the circle example, we're comparing integer counts on fixed inputs, so we used exact equality.

If you allocate an output array yourself, choose its dtype deliberately. `np.zeros(n)` creates a real array; `np.zeros(n, dtype=np.complex128)` keeps complex values. Allocating once outside a repeated update can also avoid unnecessary work.

<a id="call-c-from-python"></a>

## Call C++ from Python

**Keep Python as the driver. Move one function.**

We'll compile the same counter into a shared library, then call it using Python's `ctypes`. This is **interop**: two languages working in the same program.

This demo uses **Linux and `g++`**. The next cell checks the compiler. If it isn't available, the demo needs that setup before it can run; ask me rather than changing your runtime during the walkthrough.

In [ ]:
import ctypes
from pathlib import Path
import subprocess

compiler = subprocess.run(["g++", "--version"], check=True,
                          capture_output=True, text=True)
print(compiler.stdout.splitlines()[0])

**Write the C++ function.**

`%%writefile` saves the cell as a file in the current working directory. A row has two numbers, so row `i` starts at `2*i`. `extern "C"` gives the function a name that `ctypes` can look up.

In [ ]:
%%writefile circle.cpp
#include <cstddef>

extern "C" std::size_t count_inside(
    const double* points, std::size_t n) {
    std::size_t count = 0;
    for (std::size_t i = 0; i < n; ++i) {
        double x = points[2*i];
        double y = points[2*i + 1];
        if (x*x + y*y < 1.0) {
            ++count;
        }
    }
    return count;
}

**Compile it, then load it.**

`-O3` enables optimization. `-shared -fPIC` builds a library Python can load. Compilation happens here, before we time the call.

In [ ]:
subprocess.run([
    "g++", "-O3", "-std=c++17", "-shared", "-fPIC",
    "circle.cpp", "-o", "libcircle.so"
], check=True)

library = ctypes.CDLL(str(Path("libcircle.so").resolve()))

**Tell Python exactly what the function accepts and returns.**

Our C++ function expects a contiguous array of 64-bit real numbers, with two columns. `ndpointer` checks the dtype and memory layout; the wrapper checks the shape. A pointer alone doesn't carry that information.

In [ ]:
point_array = np.ctypeslib.ndpointer(
    dtype=np.float64, ndim=2, flags=("C_CONTIGUOUS", "ALIGNED")
)
library.count_inside.argtypes = [point_array, ctypes.c_size_t]
library.count_inside.restype = ctypes.c_size_t

def count_cpp(points):
    if points.ndim != 2 or points.shape[1] != 2:
        raise ValueError("Expected an array with shape (N, 2).")
    return library.count_inside(points, len(points))

In [ ]:
assert count_cpp(small) == 2
assert count_cpp(points) == count

cpp_seconds = median_seconds(count_cpp, points)
print(f"C++ via Python: {cpp_seconds * 1000:.3f} ms per call")

The call passes the existing NumPy storage to C++; it doesn't copy the whole array. That's why the layout checks matter. Our function only reads the input.

If you change the C++ source after loading it, **restart the kernel and rerun** before checking the rebuilt library. Loading the same filename again can reuse the library already in memory.

For your own work, this extra setup is worthwhile when you need an existing C/C++ library or a substantial compiled calculation. For a small numerical Python loop, Numba is often less work to try first.

<a id="where-julia-fits"></a>

## Where Julia fits

**Julia can keep the loop readable too.**

This is the same counter in Julia. Read it here; a Python kernel won't run Julia syntax directly.

```julia
function count_julia(points)
    count = 0
    for i in axes(points, 1)
        x = points[i, 1]
        y = points[i, 2]
        if x*x + y*y < 1.0
            count += 1
        end
    end
    return count
end

small = [0.0 0.0; 0.3 0.4; 1.0 0.0; 1.0 1.0]
@assert count_julia(small) == 2
```

Julia's ordinary arrays start at 1, so the coordinate columns are 1 and 2. `axes(points, 1)` gives the row indices. Like Numba, Julia has a first-call compilation cost; warm up a function before comparing repeated-call times.

The [earlier Julia notebook](../reference/julia_intro/julia_intro.ipynb) has more on arrays, matrix operations, and compiler output.

### Optional: call Julia from a Python notebook

For a later experiment, [JuliaCall](https://juliapy.github.io/PythonCall.jl/stable/juliacall/) lets you call Julia without switching kernels. Installing it and importing it for the first time can download Julia and set up packages. Do that before a session, not in the middle of a demo.

After installing `juliacall`, this **Python** example passes an array to a Julia function:

```python
from juliacall import Main as jl

jl.seval("sumsq(values) = sum(abs2, values)")
values = np.array([3.0, 4.0])
answer = jl.sumsq(values)
assert answer == 25.0
```

There's still a language boundary. Check the values and types you get back, and include any conversions in your timing if your real workflow needs them. Calling a compiled function on a whole array is usually more useful than calling it once per element.

<a id="your-turn"></a>

## Your turn

**Pick one change. Predict what it will do before running it.**

1. **Change the region.** Give the Python and Numba counters a radius argument. Make a small input with points inside, outside, and on the boundary. Decide the expected count yourself. Does changing the radius after compilation change the result?
2. **Change the workload.** Compare three input sizes. Where does the speedup shrink? Report the first-call cost separately from repeated calls. Explain what you included in the timing.
3. **Choose a target in your own code.** Find one calculation you'd consider speeding up. What would you time separately? What small input and independent check would you use? Which operations would you leave in NumPy or SciPy?

Be ready to show the check that supports your result. You don't need to rewrite an assignment during lab.

In [ ]:
# Try your change here. Keep the original functions above for comparison.

<a id="quick-reference"></a>

## Quick reference

| If you need to… | Start here |
| --- | --- |
| Speed up a custom numerical loop | Put it in a function with explicit inputs; try `@njit`. |
| Multiply matrices or solve a standard numerical problem | Check NumPy/SciPy's existing routines first. |
| Compare implementations | Use the same inputs; check a tiny known answer before a larger case. |
| Time compiled code | Separate setup, first call, repeated calls, and total runtime. |
| Preserve complex values | Check input, output, and preallocated array dtypes. |
| Call C/C++ | Declare argument/return types and check array shape/layout. |
| Reproduce a stochastic check | Record the seed and reuse the generated inputs when comparing code. |

The same seed in different languages doesn't promise the same random stream. For a direct comparison, pass the same samples. For a larger simulation, compare observables and uncertainties too.

**Before you keep an optimization:** What got faster? What stayed the same? What check would catch it if you changed the answer?

### More detail when you need it

- [Numba's short guide](https://numba.readthedocs.io/en/stable/user/5minguide.html) and [performance notes](https://numba.readthedocs.io/en/stable/user/performance-tips.html): supported numerical code, compilation, and timing.
- [NumPy's `ctypes` interface](https://numpy.org/doc/stable/reference/routines.ctypeslib.html) and [Python's `ctypes` documentation](https://docs.python.org/3/library/ctypes.html): array checks and function signatures.
- [Julia performance tips](https://docs.julialang.org/en/v1/manual/performance-tips/) and [JuliaCall](https://juliapy.github.io/PythonCall.jl/stable/juliacall/): compiled functions and calling Julia from Python.
- Earlier course references: [Numba](../reference/numba_ffmpeg/numba.ipynb) and [Julia](../reference/julia_intro/julia_intro.ipynb). This lab adapts their timing comparisons and circle-sampling idea. Their old lab labels are historical.